# Alpamayo–CARLA Bridge auf Google Colab prüfen

1. Lade dieses Notebook in [Colab](https://colab.research.google.com/) hoch.
2. Wähle **Laufzeit → Laufzeittyp ändern → GPU**, möglichst **L4 oder A100**, mit ausreichend Host-RAM (High-RAM).
3. Wähle unten ein Modell und führe **alle Zellen der Reihe nach** aus.

Das Notebook installiert Python 3.12 in isolierten Umgebungen. Der Colab-Kernel
bleibt unverändert; ein Kernel-Neustart ist nicht nötig. CPU-Vertragstests laufen
für **R1/1 Nano, 1.5 Nano, 2 Edge und 2 Super**, anschließend wird das ausgewählte
Modell mit einem echten NVIDIA-Datensatzclip durch die Bridge getestet.

Für echte Inferenz brauchst du Hugging-Face-Lesezugriff auf das jeweilige
[Modell](https://github.com/Jonas-a11y/alpamayo-carla-bridge/blob/main/docs/MODELS.md)
und [PhysicalAI-Autonomous-Vehicles](https://huggingface.co/datasets/nvidia/PhysicalAI-Autonomous-Vehicles).
Akzeptiere dort eventuell erforderliche Bedingungen. Lege optional ein Colab-Secret
**HF_TOKEN** an und erlaube diesem Notebook den Zugriff; sonst fragt eine Zelle
den Token verdeckt ab. Schreibe ihn nicht in eine Codezelle.

Edge benötigt etwa ≥16 GB VRAM mit BF16, Nano ≥24 GB. **T4 bietet keine native
BF16-Unterstützung. Super (34B) passt üblicherweise nicht auf eine Colab-GPU**;
seine Adaptertests laufen trotzdem. Fehlende Ressourcen werden als **SKIPPED**
berichtet, fehlgeschlagene Tests als **FAILED**. Ein übersprungener GPU-Lauf gilt
nicht als erfolgreicher Inferenztest.

Optional lädt und startet das Notebook **CARLA 0.9.16 direkt in Colab** und
führt eine kurze echte Fahrt aus. Dafür `RUN_CARLA_DRIVE=True` setzen.
Ein externer Server ist ebenfalls möglich (`START_CARLA_IN_COLAB=False`).
Datensatz-Inferenz prüft Gewichte/API/Trajektorien; Fahrqualität braucht zusätzlich
CARLA und kalibrierte Kameras. Installationen und erste Downloads können dauern.

In [ ]:
VARIANT = "edge"  # @param ["edge", "r1", "r15", "super"]
RUN_GPU_INFERENCE = True  # @param {type:"boolean"}
RUN_CARLA_DRIVE = True  # @param {type:"boolean"}
START_CARLA_IN_COLAB = True  # @param {type:"boolean"}
RENDER_VIDEO = True  # @param {type:"boolean"}
CARLA_HOST = ""  # @param {type:"string"}
CARLA_PORT = 2000  # @param {type:"integer"}
MAX_STEPS = 30  # @param {type:"integer"}
NAV_TEXT = "Follow the current lane."  # @param {type:"string"}
BRIDGE_REF = "main"  # @param {type:"string"}

# Für einen reproduzierbaren Wiederholungslauf hier den Commit aus summary.json einsetzen.
assert VARIANT in {"edge", "r1", "r15", "super"}
assert MAX_STEPS > 0
if START_CARLA_IN_COLAB:
    CARLA_HOST = "127.0.0.1"
assert not RUN_CARLA_DRIVE or CARLA_HOST.strip(), "Setze die erreichbare CARLA-Serveradresse."
assert VARIANT != "edge" or NAV_TEXT.strip(), "Setze deine Navigationsanweisung."

## 1. Repository und Werkzeuge

Alle Dateien liegen unter `/content/alpamayo-bridge-check`. Beim erneuten Ausführen
werden die vorhandenen Umgebungen verwendet. Der Bericht hält den tatsächlich
geprüften Bridge-Commit fest.

In [ ]:
from pathlib import Path
import getpass
import json
import os
import shutil
import subprocess
import sys
import time

ROOT = Path("/content/alpamayo-bridge-check")
REPO = ROOT / "repo"
REPORT = ROOT / f"report-{time.time_ns()}"
REPORT.mkdir(parents=True, exist_ok=True)
RESULTS = {}
HF_TOKEN = None

def run_check(name, command, *, cwd=None, env=None, required=True):
    print(f"\n{name} ...", flush=True)
    started = time.monotonic()
    # Colabs Inline-Backend ist in den isolierten Umgebungen nicht installiert.
    child_env = (os.environ if env is None else env).copy()
    child_env["MPLBACKEND"] = "Agg"
    result = subprocess.run(command, cwd=cwd, env=child_env, capture_output=True, text=True)
    log = result.stdout + result.stderr
    if HF_TOKEN:
        log = log.replace(HF_TOKEN, "[redacted]")
    (REPORT / f"{name}.log").write_text(log)
    print(log[-12000:])
    RESULTS[name] = {"status": "PASSED" if result.returncode == 0 else "FAILED",
                     "returncode": result.returncode,
                     "seconds": round(time.monotonic() - started, 2)}
    if required and result.returncode:
        raise RuntimeError(f"{name} fehlgeschlagen. Siehe Ausgabe und Logs unter {REPORT}")
    return result

if shutil.which("uv") is None:
    run_check("install-uv", [sys.executable, "-m", "pip", "install", "uv"])
UV = shutil.which("uv")
run_check("python312", [UV, "python", "install", "3.12"])
if not (REPO / ".git").exists():
    run_check("clone", ["git", "clone", "--filter=blob:none",
                        "https://github.com/Jonas-a11y/alpamayo-carla-bridge.git", str(REPO)])
run_check("fetch", ["git", "fetch", "origin", BRIDGE_REF], cwd=REPO)
run_check("checkout", ["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO)
BRIDGE_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
print("Geprüfter Bridge-Commit:", BRIDGE_SHA)

ENV_PYTHON = {}
for runtime in ("core", "edge", "nano-super"):
    folder = ROOT / runtime
    if not (folder / "bin/python").exists():
        run_check(f"venv-{runtime}", [UV, "venv", "--python", "3.12", str(folder)])
    ENV_PYTHON[runtime] = str(folder / "bin/python")

## 2. Beide Modellumgebungen installieren

Die PyTorch-CUDA-Wheels werden von der offiziellen Quelle installiert. Edge nutzt
Transformers 5.17.0, Nano/Super 4.57.1. Die drei Nano/Super-Quellpakete werden nach
ihren gemeinsamen Inferenzabhängigkeiten mit `--no-deps` installiert, damit keine
unnötige FlashAttention-Kompilierung startet. CARLA-Client und Video-Abhängigkeiten
sind enthalten; Modellgewichte werden erst beim optionalen GPU-Test geladen.

In [ ]:
for runtime in ("edge", "nano-super"):
    python = ENV_PYTHON[runtime]
    run_check(f"torch-{runtime}", [UV, "pip", "install", "--python", python,
        "torch==2.8.0", "torchvision==0.23.0",
        "--index-url", "https://download.pytorch.org/whl/cu126"])
    run_check(f"runtime-{runtime}", [UV, "pip", "install", "--python", python,
        "-e", f"{REPO}[{runtime},test,carla,video]"])

NATIVE_SOURCES = [
    "alpamayo_r1 @ git+https://github.com/NVlabs/alpamayo.git@11a0e01c13a5622377c45ee37d653351453ec43b",
    "alpamayo1_5 @ git+https://github.com/NVlabs/alpamayo1.5.git@36aeb4c5938cbc2eb2aed33b22434773da4ab639",
    "alpamayo2_super @ git+https://github.com/NVlabs/alpamayo2.git@59776920bbd981f2c989213bba5ee8594e71ed1d",
]
run_check("native-sources", [UV, "pip", "install", "--python", ENV_PYTHON["nano-super"],
                              "--no-deps", *NATIVE_SOURCES])
for runtime in ("edge", "nano-super"):
    run_check(f"versions-{runtime}", [UV, "pip", "freeze", "--python", ENV_PYTHON[runtime]])
if shutil.which("ffmpeg") is None:
    run_check("ffmpeg-index", ["apt-get", "update", "-qq"])
    run_check("ffmpeg-install", ["apt-get", "install", "-y", "-qq", "ffmpeg"])


## 3. Alle Adapter und die installierbare Bridge prüfen

Die Tests verwenden echte NVIDIA-Importe/Input-/Prompt-Helfer, simulierte
Modellvorhersagen und eine simulierte CARLA-Welt. Sie prüfen auch Kamera-/Pose-
Synchronisation, Koordinaten, Modellwahl, Neustart, Bremsen und Cleanup.
Die Wheel-Prüfung importiert ausschließlich den gebauten Core aus einem anderen
Arbeitsverzeichnis. Beide Testsuiten laufen auch dann, wenn eine davon fehlschlägt.

In [ ]:
for runtime, marker in (("edge", "not nano_super"), ("nano-super", "not edge")):
    run_check(f"tests-{runtime}", [ENV_PYTHON[runtime], "-m", "pytest", "-q", "-m", marker,
        f"--junitxml={REPORT / (runtime + '.xml')}"], cwd=REPO, required=False)

run_check("build-wheel", [UV, "build", str(REPO), "--wheel", "--out-dir", str(REPORT / "wheel")])
wheel = next((REPORT / "wheel").glob("alpamayo_carla_bridge-*.whl"))
run_check("install-wheel", [UV, "pip", "install", "--python", ENV_PYTHON["core"], str(wheel)])
WHEEL_CHECK = r'''
import subprocess, sys
import bridge
from bridge import CarlaBridge, AlpamayoInference, ObservationBuffer
assert not {'torch', 'transformers', 'carla', 'PIL', 'matplotlib'} & sys.modules.keys()
assert 'site-packages' in bridge.__file__, bridge.__file__
result = subprocess.run([sys.executable, '-m', 'bridge.driver', '--help'], capture_output=True, text=True)
assert result.returncode == 0, result.stderr
assert '--model {r1,r15,edge,super}' in result.stdout
print('Core-Wheel importierbar; CLI und vier Varianten vorhanden; optionale Runtimes nicht geladen.')
'''
run_check("wheel-import", [ENV_PYTHON["core"], "-c", WHEEL_CHECK], cwd=ROOT, required=False)
CONTRACTS_OK = all(RESULTS[name]["status"] == "PASSED"
                   for name in ("tests-edge", "tests-nano-super", "wheel-import"))

## 4. GPU und freien Speicher prüfen

Der Test prüft CUDA/BF16 sowie freien GPU-/Host-/Datenträgerspeicher, bevor Gewichte
oder Datensatz geladen werden. Die Grenzen sind Vorprüfungen, keine Garantie,
dass jede GPU-/Treiberkombination passt. Bei fehlenden Ressourcen oder roten
Vertragstests werden echte Inferenz und CARLA übersprungen.

In [ ]:
SELECTED_PYTHON = ENV_PYTHON["edge" if VARIANT == "edge" else "nano-super"]
SETTINGS = {"variant": VARIANT, "nav_text": NAV_TEXT,
            "carla_host": CARLA_HOST, "carla_port": CARLA_PORT, "max_steps": MAX_STEPS}
(ROOT / "settings.json").write_text(json.dumps(SETTINGS))
GPU_PROBE = r'''
import json, shutil, sys
from pathlib import Path
import torch
root = Path(sys.argv[1])
report = Path(sys.argv[2])
variant = json.loads((root / 'settings.json').read_text())['variant']
info = {'cuda': torch.cuda.is_available(), 'disk_free_gb': shutil.disk_usage(root).free / 1e9}
reasons = []
if not info['cuda']:
    reasons.append('CUDA nicht verfügbar; GPU-Laufzeit und Treiber prüfen')
else:
    props = torch.cuda.get_device_properties(0)
    info.update(gpu=props.name, vram_gb=props.total_memory / 1e9,
                free_vram_gb=torch.cuda.mem_get_info()[0] / 1e9,
                bf16=torch.cuda.is_bf16_supported() and props.major >= 8,
                torch=torch.__version__, cuda_runtime=torch.version.cuda)
    if not info['bf16']:
        reasons.append('keine native BF16-Unterstützung')
    if info['free_vram_gb'] < {'edge': 16, 'r1': 24, 'r15': 24, 'super': 80}[variant]:
        reasons.append('zu wenig freier VRAM für die gewählte Referenzkonfiguration')
if info['cuda'] and variant in ('edge', 'r1', 'r15'):
    memory = dict(line.split(':', 1) for line in Path('/proc/meminfo').read_text().splitlines())
    info['host_available_ram_gb'] = int(memory['MemAvailable'].split()[0]) * 1024 / 1e9
    if info['host_available_ram_gb'] < (12 if variant == 'edge' else 24):
        reasons.append('zu wenig freier Host-RAM; High-RAM-Laufzeit wählen')
if info['disk_free_gb'] < {'edge': 20, 'r1': 40, 'r15': 40, 'super': 90}[variant]:
    reasons.append('zu wenig freier Datenträgerspeicher für Gewichte und Sample-Cache')
info.update(ready=not reasons, reasons=reasons)
(report / 'gpu.json').write_text(json.dumps(info, indent=2))
print(json.dumps(info, indent=2))
'''
run_check("gpu-preflight", [SELECTED_PYTHON, "-c", GPU_PROBE, str(ROOT), str(REPORT)])
GPU_INFO = json.loads((REPORT / "gpu.json").read_text())
GPU_READY = GPU_INFO["ready"] and CONTRACTS_OK
if not GPU_READY:
    print("SKIPPED: echter GPU-/CARLA-Lauf:", GPU_INFO["reasons"] or ["Vertragstests fehlgeschlagen"])

## 5. Hugging-Face-Zugriff und Datensatz-SDK

Nur bei angefordertem und möglich gewordenen GPU-/CARLA-Lauf wird nach dem Token
gefragt. Er bleibt im Speicher und wird als Umgebungsvariable an den Testprozess
übergeben. Der Bericht enthält weder den Token noch den Hugging-Face-Cache.

In [ ]:
AUTH_ENV = os.environ.copy()
if GPU_READY and (RUN_GPU_INFERENCE or RUN_CARLA_DRIVE):
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = getpass.getpass("Hugging-Face READ-Token (verdeckt): ")
    if not HF_TOKEN or not HF_TOKEN.strip():
        raise ValueError("Ein Hugging-Face-Lesetoken wird für diesen Lauf benötigt.")
    HF_TOKEN = HF_TOKEN.strip()
    AUTH_ENV["HF_TOKEN"] = HF_TOKEN
    AUTH_ENV["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
    if RUN_GPU_INFERENCE:
        run_check("dataset-sdk", [UV, "pip", "install", "--python", SELECTED_PYTHON,
                                   "physical-ai-av==0.2.0"])

## 6. Echte Modellinferenz durch die Bridge

Ein offizieller Beispielclip wird mit dem passenden NVIDIA-Loader gelesen.
Die Bridge bekommt ausschließlich Bilder, Kamera-IDs, Historie und gegebenenfalls
Navigation; zukünftige Ground-Truth-Posen werden nur nach der Vorhersage zum
Vergleich verwendet. Für Super wird aus dem kanonischen Datensatzring zuerst das
native Sechs-Kamera-Profil ausgewählt. Pro Lauf wird **ein Modell** mit **einem
Trajektorienkandidaten** getestet; wähle eine andere Variante für einen weiteren Lauf.

Ergebnisse: 64 endliche XY-Wegpunkte, Reasoning, Zeit-/Speicherwerte und eine PNG-
Trajektoriengrafik sowie das Front-Weitwinkelbild bei t0. Ein einzelner Clip oder dessen Abweichung belegt keine Fahrqualität.

In [ ]:
GPU_SMOKE = r'''
import importlib, json, sys, time
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from bridge import AlpamayoInference, load_config

root = Path(sys.argv[1])
report = Path(sys.argv[2])
repo = root / 'repo'
settings = json.loads((root / 'settings.json').read_text())
variant = settings['variant']
packages = {'edge': 'alpamayo2_edge', 'r1': 'alpamayo_r1',
            'r15': 'alpamayo1_5', 'super': 'alpamayo2_super'}
loader = importlib.import_module(packages[variant] + '.load_physical_aiavdataset')
clip_id, t0_us = '030c760c-ae38-49aa-9ad8-f5650a545d26', 5_100_000
kwargs = {'nav_text': settings['nav_text']} if variant == 'edge' else {}
if variant == 'super':
    kwargs['include_calibration'] = False
print('Lade echten NVIDIA-Clip:', clip_id, flush=True)
data = loader.load_physical_aiavdataset(clip_id, t0_us=t0_us, **kwargs)
if variant == 'super':
    from alpamayo2_super.input_profiles import select_task_input
    data = select_task_input(data, 'trajectory')
keys = ['image_frames', 'camera_indices', 'ego_history_xyz', 'ego_history_rot']
if variant == 'edge':
    keys += ['absolute_timestamps', 'nav_text']
sample = {key: data[key] for key in keys}
if variant == 'r15':
    sample['nav_text'] = settings['nav_text']
assert not any(key.startswith('ego_future') for key in sample)
config = load_config(repo / 'config.yaml', model=variant)
config['alpamayo'].update(device='cuda:0', dtype='bfloat16', num_traj_samples=1,
                          attn_implementation='sdpa')
inference = AlpamayoInference(config)
print('Lade Modellgewichte:', variant, flush=True)
started = time.monotonic()
inference.load()
torch.cuda.synchronize()
load_seconds = time.monotonic() - started
torch.manual_seed(42)
torch.cuda.reset_peak_memory_stats()
started = time.monotonic()
output = inference.predict(sample)
torch.cuda.synchronize()
seconds = time.monotonic() - started
xy = np.asarray(output.trajectory)
assert xy.shape == (64, 2) and np.isfinite(xy).all()
result = {'variant': variant, 'clip_id': clip_id, 't0_us': t0_us,
          'trajectory': output.trajectory, 'reasoning': output.reasoning,
          'load_seconds': load_seconds, 'inference_seconds': seconds,
          'peak_allocated_gb': torch.cuda.max_memory_allocated() / 1e9,
          'image_shape': list(sample['image_frames'].shape),
          'camera_ids': sample['camera_indices'].cpu().tolist()}
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(xy[:, 0], xy[:, 1], label='Modellvorhersage')
if 'ego_future_xyz' in data:
    gt = data['ego_future_xyz'][0, 0, :, :2].cpu().numpy()
    result['mean_displacement_error_m'] = float(np.linalg.norm(xy - gt, axis=1).mean())
    ax.plot(gt[:, 0], gt[:, 1], '--', label='Datensatz-Trajektorie')
ax.scatter([0], [0], label='Ego jetzt')
ax.set(xlabel='X vorwärts (m)', ylabel='Y links (m)', title=f'{variant}: ein Beispielclip')
ax.axis('equal')
ax.grid(True)
ax.legend()
fig.tight_layout()
fig.savefig(report / 'trajectory.png', dpi=140)
plt.close(fig)
# Letztes front_wide-Bild bei t0; Edge hat einen Eintrag pro Kamerazeitpunkt.
front_index = torch.where(sample['camera_indices'] == 1)[0][-1].item()
front = sample['image_frames'][front_index, -1].cpu()
if front.is_floating_point():
    front = (front.clamp(0, 1) * 255).round().to(torch.uint8)
snapshot = Image.fromarray(front.permute(1, 2, 0).numpy())
snapshot.thumbnail((960, 540))
snapshot.save(report / 'front_wide.jpg')
(report / 'prediction.json').write_text(json.dumps(result, indent=2))
print('64 endliche Wegpunkte; Reasoning:', output.reasoning)
print('Inferenz (s):', round(seconds, 2), 'Peak GPU (GB):', round(result['peak_allocated_gb'], 2))
'''
if RUN_GPU_INFERENCE and GPU_READY:
    run_check("gpu-inference", [SELECTED_PYTHON, "-c", GPU_SMOKE, str(ROOT), str(REPORT)],
              env=AUTH_ENV, required=False)
    if RESULTS["gpu-inference"]["status"] == "PASSED":
        from IPython.display import Image, display
        display(Image(filename=str(REPORT / "trajectory.png")))
else:
    RESULTS["gpu-inference"] = {"status": "SKIPPED", "reason":
        "deaktiviert" if not RUN_GPU_INFERENCE else GPU_INFO["reasons"] or ["Vertragstests fehlgeschlagen"]}

## 7. Optional: CARLA direkt in Colab starten und fahren

Setze `RUN_CARLA_DRIVE=True`; `START_CARLA_IN_COLAB=True` ist bereits gewählt.
`CARLA_HOST` wird automatisch auf `127.0.0.1` gesetzt. Du brauchst keinen eigenen
Server, keine öffentliche Adresse und keinen Tunnel.

Das Notebook lädt die [offizielle CARLA-0.9.16-Linux-Version](https://github.com/carla-simulator/carla/releases/tag/0.9.16)
(ca. 8,35 GB Download; mindestens 35 GB freier Platz vor der Installation),
installiert Laufzeitbibliotheken und startet Unreal als separaten Benutzer mit
`-RenderOffScreen -vulkan -nosound -quality-level=Low`. Kameras rendern weiterhin
Bilder. Falls Colab nur CUDA-Bibliotheken bereitstellt, wird das passende
offizielle NVIDIA-Treiberarchiv ausschließlich für seine Grafikbibliotheken
entpackt; der Kernel-Treiber wird nicht neu installiert.

Der Worker prüft NVIDIA/Vulkan, RPC/Version und ein echtes RGB-Kamerabild. CARLA
teilt sich die GPU mit Alpamayo; nach dem Serverstart wird der freie VRAM erneut
geprüft. Eine erfolgreiche Datensatz-Inferenz liefert den gemessenen GPU-Peak
für die Vorprüfung (plus 2 GB Reserve); sonst gelten die Modell-Richtwerte.
L4 kann knapp werden, A100 bietet mehr Reserve. Manche Colab-Laufzeiten
exportieren keine nutzbare Vulkan-GPU: Das wird mit Logs als **FAILED** berichtet.
Der erste Colab-Lauf scheiterte am GLX-ICD. Der Worker nutzt jetzt
[NVIDIAs dokumentierten EGL-ICD für Umgebungen ohne X11](https://download.nvidia.com/XFree86/Linux-x86_64/580.82.07/README/installedcomponents.html).
Dieser korrigierte Startmodus braucht noch einen erneuten Colab-Lauf.

Nach der Fahrt oder einem Fehler stoppt der Worker seine eigene Serverprozessgruppe.
Downloads bleiben für Wiederholungsläufe erhalten; Server- und Vulkan-Logs liegen
im Bericht. Bei einem kompletten Laufzeitabbruch setzt Colab die VM zurück.

Für einen externen **CARLA-0.9.16-Server** deaktiviere `START_CARLA_IN_COLAB` und
setze die von Colab erreichbare `CARLA_HOST`-Adresse. Der Server braucht RGB-Rendering
und eine dedizierte Welt, die nur dieser Client tickt. Die Bridge kann die Map
laden und spawnt ein Fahrzeug. Kamera-/Achskalibrierung und Fahrevaluation bleiben
zusätzliche Aufgaben. Mit `RENDER_VIDEO=True` wird die Aufnahme anschließend
komponiert. Bei 10 Hz ergeben `MAX_STEPS=30` etwa drei Sekunden Fahrt; der
Standardwert 30 liefert drei Sekunden. Mehr Schritte bedeuten mehr Modellaufrufe.


In [ ]:
CARLA_SMOKE = r'''
import json, sys
from pathlib import Path
from bridge import CarlaBridge, load_config
root = Path(sys.argv[1])
report = Path(sys.argv[2])
settings = json.loads((root / 'settings.json').read_text())
config = load_config(root / 'repo/config.yaml', model=settings['variant'])
config['carla'].update(host=settings['carla_host'], port=settings['carla_port'])
config['alpamayo']['device'] = 'cuda:0'
if settings['variant'] in ('edge', 'r15'):
    config['alpamayo']['nav_text'] = settings['nav_text']
config['recording']['enabled'] = True
with CarlaBridge(config, report / 'recordings') as bridge:
    for step in range(settings['max_steps']):
        result = bridge.step()
        assert len(result.output.trajectory) == 64
        print('Schritt', step + 1, 'Frame', result.state.frame, 'Control', result.command.to_dict())
    session = str(bridge.session_dir.relative_to(report))
(report / 'carla.json').write_text(json.dumps({'steps': settings['max_steps'], 'session': session}))
print('Fahrt abgeschlossen; Bridge-Cleanup aufgerufen; Aufnahme:', session)
'''
CARLA_LOCAL = r'''
import ctypes.util
import json
import os
from pathlib import Path
import pwd
from queue import Queue
import shutil
import signal
import subprocess
import sys
import time

import carla
import numpy as np
from PIL import Image
import torch


def wait_for_server(process, client, timeout=180):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        if process.poll() is not None:
            raise RuntimeError(f'CARLA exited before RPC readiness (rc={process.returncode}); see carla-server.log')
        try:
            version = client.get_server_version()
            if not version.startswith('0.9.16'):
                raise ValueError(f'Unexpected CARLA server version: {version}')
            return client.get_world()
        except RuntimeError:
            time.sleep(1)
    raise TimeoutError(f'CARLA RPC not ready after {timeout} s; see carla-server.log')


def stop_server(process):
    # Only the process group created by this worker; includes the Unreal child.
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        process.wait()
        return
    try:
        process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        pass
    try:
        os.killpg(process.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    process.wait()


root = Path(sys.argv[1])
report = Path(sys.argv[2])
settings = json.loads((root / 'settings.json').read_text())
server = root / 'carla-0.9.16'
info = {'status': 'FAILED', 'version': '0.9.16', 'host': '127.0.0.1', 'port': settings['carla_port']}
process = None
try:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'libvulkan1', 'vulkan-tools',
                    'libsdl2-2.0-0', 'libomp5', 'xdg-user-dirs', 'libnss3', 'libglu1-mesa', 'libegl1', 'libopengl0'], check=True)
    try:
        user = pwd.getpwnam('carla-colab')
    except KeyError:
        subprocess.run(['useradd', '--create-home', '--shell', '/bin/bash', 'carla-colab'], check=True)
        user = pwd.getpwnam('carla-colab')
    subprocess.run(['usermod', '-a', '-G', 'video', user.pw_name], check=True)
    runtime = Path(user.pw_dir) / 'runtime'
    runtime.mkdir(mode=0o700, exist_ok=True)
    os.chown(runtime, user.pw_uid, user.pw_gid)
    env = os.environ.copy()
    env.pop('SDL_VIDEODRIVER', None)
    env.pop('HF_TOKEN', None)
    env.update(HOME=user.pw_dir, XDG_RUNTIME_DIR=str(runtime))
    # Colab often exports only CUDA. Extract matching user-space graphics libs;
    # never run nvidia-installer or replace the host kernel driver.
    # NVIDIA recommends EGL for a Vulkan ICD without X11 client libraries.
    icd = Path(user.pw_dir) / 'nvidia_icd.json'
    api_version = '1.2.0'
    library = ctypes.util.find_library('EGL_nvidia')
    as_user = ['runuser', '-u', user.pw_name, '--']
    if not library:
        driver = subprocess.check_output(['nvidia-smi', '--query-gpu=driver_version',
                                          '--format=csv,noheader'], text=True).splitlines()[0].strip()
        if not all(part.isdigit() for part in driver.split('.')):
            raise ValueError(f'Unexpected NVIDIA driver version: {driver}')
        graphics = root / f'nvidia-graphics-{driver}'
        library = str(graphics / f'libEGL_nvidia.so.{driver}')
        if not Path(library).exists():
            runfile = root / f'NVIDIA-{driver}.run'
            url = f'https://us.download.nvidia.com/tesla/{driver}/NVIDIA-Linux-x86_64-{driver}.run'
            subprocess.run(['curl', '--fail', '--location', '--retry', '3', '--output', str(runfile), url], check=True)
            subprocess.run(['/bin/sh', str(runfile), '--extract-only', '--target', str(graphics)], check=True)
            subprocess.run(['chown', '-R', f'{user.pw_uid}:{user.pw_gid}', str(graphics)], check=True)
            runfile.unlink()
        if not Path(library).exists():
            raise RuntimeError('Matching NVIDIA runfile contains no graphics library; see carla-drive.log.')
        descriptor = graphics / 'nvidia_icd.json'
        if descriptor.exists():
            api_version = json.loads(descriptor.read_text())['ICD']['api_version']
        as_user += ['env', f"LD_LIBRARY_PATH={graphics}:{env.get('LD_LIBRARY_PATH', '')}"]
    icd.write_text(json.dumps({'file_format_version': '1.0.0',
                              'ICD': {'library_path': library, 'api_version': api_version}}))
    env['VK_ICD_FILENAMES'] = str(icd)
    info.update(vulkan_icd=library, vulkan_api_version=api_version)
    access = subprocess.run([*as_user, 'nvidia-smi', '-L'], env=env,
                            capture_output=True, text=True)
    vulkan = subprocess.run([*as_user, 'vulkaninfo', '--summary'],
                            env=env, capture_output=True, text=True)
    (report / 'vulkan.log').write_text(access.stdout + access.stderr + vulkan.stdout + vulkan.stderr)
    if access.returncode or vulkan.returncode or not any('deviceName' in line and 'NVIDIA' in line
                                   for line in vulkan.stdout.splitlines()):
        raise RuntimeError('No NVIDIA Vulkan device available. CUDA alone is insufficient for CARLA RGB; see vulkan.log.')
    client = carla.Client('127.0.0.1', settings['carla_port'])
    client.set_timeout(2)
    try:
        client.get_server_version()
    except RuntimeError:
        pass
    else:
        raise RuntimeError('CARLA port already in use; stop the previous server or select another CARLA_PORT.')
    if not (server / '.installed').exists():
        if shutil.disk_usage(root).free < 35e9:
            raise RuntimeError('Need at least 35 GB free disk for CARLA download and extraction.')
        archive = root / 'CARLA_0.9.16.tar.gz'
        partial = archive.with_suffix('.partial')
        url = 'https://downloads.carlasim.com/Linux/CARLA_0.9.16.tar.gz'
        print('Downloading official CARLA 0.9.16 archive (~8.35 GB)', flush=True)
        subprocess.run(['curl', '--fail', '--location', '--retry', '3', '--output', str(partial), url], check=True)
        partial.replace(archive)
        server.mkdir(exist_ok=True)
        subprocess.run(['tar', '-xzf', str(archive), '-C', str(server), '--no-same-owner'], check=True)
        subprocess.run(['chown', '-R', f'{user.pw_uid}:{user.pw_gid}', str(server)], check=True)
        (server / '.installed').touch()
        archive.unlink()
    command = [*as_user, '/bin/bash', str(server / 'CarlaUE4.sh'),
               '-RenderOffScreen', '-vulkan', '-nosound', '-quality-level=Low',
               f"-carla-rpc-port={settings['carla_port']}"]
    with (report / 'carla-server.log').open('w') as log:
        process = subprocess.Popen(command, cwd=server, env=env, stdout=log,
                                   stderr=subprocess.STDOUT, start_new_session=True)
        world = wait_for_server(process, client)
        client.set_timeout(30)
        blueprint = world.get_blueprint_library().find('sensor.camera.rgb')
        blueprint.set_attribute('image_size_x', '64')
        blueprint.set_attribute('image_size_y', '64')
        transform = world.get_map().get_spawn_points()[0]
        transform.location.z += 1.6
        camera = world.spawn_actor(blueprint, transform)
        try:
            queue = Queue()
            camera.listen(queue.put)
            frame = queue.get(timeout=30)
            rgb = np.frombuffer(frame.raw_data, dtype=np.uint8).reshape(64, 64, 4)[:, :, :3][:, :, ::-1]
            if np.ptp(rgb) == 0:
                raise RuntimeError('CARLA RGB probe returned a uniform/empty image.')
            Image.fromarray(rgb).save(report / 'carla-render-probe.jpg')
        finally:
            camera.stop()
            camera.destroy()
        info.update(status='PASSED', rgb_probe=True)
        free_gb = torch.cuda.mem_get_info()[0] / 1e9
        info['free_vram_after_start_gb'] = free_gb
        required_gb = {'edge': 16, 'r1': 24, 'r15': 24, 'super': 80}[settings['variant']]
        prediction_file = report / 'prediction.json'
        if prediction_file.exists():
            prediction = json.loads(prediction_file.read_text())
            if prediction['variant'] == settings['variant']:
                required_gb = prediction['peak_allocated_gb'] + 2
        info['required_free_vram_gb'] = required_gb
        if free_gb < required_gb:
            raise RuntimeError(f'CARLA started, but only {free_gb:.1f} GB VRAM remains; {required_gb} GB needed for this model.')
        subprocess.run([sys.executable, str(root / 'carla_smoke.py'), str(root), str(report)], check=True)
except BaseException as error:
    info['error'] = str(error)
    raise
finally:
    if process is not None:
        stop_server(process)
    (report / 'carla-server.json').write_text(json.dumps(info, indent=2))
'''
if RUN_CARLA_DRIVE and GPU_READY:
    if START_CARLA_IN_COLAB:
        (ROOT / "carla_smoke.py").write_text(CARLA_SMOKE)
    run_check("carla-drive", [SELECTED_PYTHON, "-c",
              CARLA_LOCAL if START_CARLA_IN_COLAB else CARLA_SMOKE, str(ROOT), str(REPORT)],
              env=AUTH_ENV, required=False)
    if START_CARLA_IN_COLAB:
        server_report = REPORT / "carla-server.json"
        RESULTS["carla-server"] = (json.loads(server_report.read_text()) if server_report.exists() else
            {"status": "FAILED", "reason": "Server-Worker abgebrochen; carla-drive.log prüfen"})
else:
    RESULTS["carla-drive"] = {"status": "SKIPPED", "reason":
        "deaktiviert" if not RUN_CARLA_DRIVE else GPU_INFO["reasons"] or ["Vertragstests fehlgeschlagen"]}
RESULTS.setdefault("carla-server", {"status": "SKIPPED", "reason":
    "externer Server gewählt" if RUN_CARLA_DRIVE and not START_CARLA_IN_COLAB else
    "Fahrt deaktiviert oder GPU-/Vertragstests nicht bereit"})


## 8. CARLA-Fahrvideo mit vorhergesagtem Pfad

Nach erfolgreicher Fahrt wird die aufgezeichnete Kamerasequenz als H.264-MP4
komponiert. Front-Weitwinkel und Teleansicht zeigen den vorhergesagten Zukunftspfad
in Cyan direkt im Kamerabild; zusätzlich bleiben Trajektorienkarte und Reasoning
sichtbar. Die Projektion nutzt Sichtfeld, Kameraposition/-ausrichtung und Ego-Ursprung
aus der Aufnahme. Da die Bridge XY-Pfade liefert, wird die Straße als aktuelle
Ego-Ebene angenähert; Kalibrierung, Steigungen und Unebenheiten beeinflussen die
Einblendung.

Fahrt und Video sind bereits aktiviert. Der Standardwert MAX_STEPS=30 liefert
etwa drei Sekunden Simulationszeit (10 Bilder/s). Auf deiner L4 dauerte die
bestätigte Datensatz-Inferenz rund fünf Sekunden; 30 Fahrentscheidungen können
daher mehrere Minuten Rechenzeit brauchen.

Das Video erscheint hier und liegt als carla-drive.mp4 mit video.json im ZIP.
Wenn CARLA scheitert, wird kein Ersatzvideo aus einem Standbild erzeugt.
Der Bericht zeigt dann video-render: SKIPPED mit dem Grund; Vorhersage und PNG-
Grafik aus einer erfolgreichen Datensatz-Inferenz bleiben separat erhalten.


In [ ]:
VIDEO_RENDER = r'''
import json, shutil, sys
from pathlib import Path
from bridge.video_composer import VideoComposer

report = Path(sys.argv[1])
carla = json.loads((report / 'carla.json').read_text())
config = {'recording': {'video_width': 1280, 'video_height': 720, 'fps': 10},
          'run': {'max_steps': carla['steps']}}
output = report / 'carla-drive.mp4'
shutil.copyfile(VideoComposer(report / carla['session'], config).compose(), output)
info = {'source': 'carla', 'frames': carla['steps'], 'fps': 10,
        'seconds': carla['steps'] / 10, 'file': output.name,
        'description': 'Recorded CARLA camera frames and telemetry',
        'overlay': 'Predicted XY future in front-wide and tele cameras; flat ego plane'}
(report / 'video.json').write_text(json.dumps(info, indent=2))
print(json.dumps(info, indent=2))
'''
if not RENDER_VIDEO:
    RESULTS['video-render'] = {'status': 'SKIPPED', 'reason': 'deaktiviert'}
elif RESULTS.get('carla-drive', {}).get('status') != 'PASSED':
    RESULTS['video-render'] = {'status': 'SKIPPED',
        'reason': 'Keine erfolgreiche CARLA-Fahrt; carla-drive.log prüfen'}
    print('Kein Fahrvideo: CARLA-Fahrt fehlgeschlagen oder deaktiviert. Siehe carla-drive.log.')
else:
    run_check('video-render', [SELECTED_PYTHON, '-c', VIDEO_RENDER, str(REPORT)], required=False)
    if RESULTS['video-render']['status'] == 'PASSED':
        from IPython.display import Video, display
        display(Video(str(REPORT / 'carla-drive.mp4'), embed=True))


## 9. Bericht herunterladen

Das ZIP enthält Status/Commit/GPU-Angaben, Testlogs und JUnit-Ergebnisse sowie
gegebenenfalls Vorhersage, Grafik, Videovorschau und CARLA-Aufnahmen. Es enthält weder Secrets
noch Gewichte/Caches. Du kannst mir den Bericht anschließend zur Auswertung geben.
Wenn eine Installationszelle abgebrochen ist, führe diese letzte Zelle trotzdem aus,
um die bis dahin vorhandenen Logs zu exportieren.

In [ ]:
for name in ("tests-edge", "tests-nano-super", "wheel-import", "gpu-inference", "carla-drive", "carla-server", "video-render"):
    RESULTS.setdefault(name, {"status": "SKIPPED", "reason": "nicht erreicht; vorherige Zelle prüfen"})
summary = {"bridge_commit": globals().get("BRIDGE_SHA"), "selected_variant": VARIANT,
           "gpu": globals().get("GPU_INFO"), "checks": RESULTS}
(REPORT / "summary.json").write_text(json.dumps(summary, indent=2))
for name, result in RESULTS.items():
    print(f"{result['status']:7} {name}")
print("SKIPPED ist kein bestandener Inferenz-/Fahrtest.")
archive = shutil.make_archive(str(ROOT / "alpamayo-colab-report"), "zip", REPORT)
from google.colab import files
files.download(archive)